# 05.2 — MoE Inference: Routing, Load Balancing & Scaling Economics

Mixture-of-Experts models activate only a subset of parameters per token, enabling massive capacity with sub-linear compute. This lab covers:
1. Top-K routing simulation
2. Expert load imbalance measurement
3. Double penalty calculator (compute + memory waste)
4. Wide-EP communication volume modeling
5. MoE vs dense cost comparison at various GPU counts

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from content.utils.benchmark import Timer

## 1. MoE Routing Simulation (Top-K Gating)

In [ ]:
def top_k_routing(gate_logits: np.ndarray, k: int = 2):
    """Simulate top-k expert routing."""
    probs = np.exp(gate_logits) / np.exp(gate_logits).sum(axis=1, keepdims=True)
    top_k_indices = np.argsort(probs, axis=1)[:, -k:]
    top_k_weights = np.take_along_axis(probs, top_k_indices, axis=1)
    top_k_weights = top_k_weights / top_k_weights.sum(axis=1, keepdims=True)
    return top_k_indices, top_k_weights

np.random.seed(42)
num_tokens, num_experts, top_k = 1024, 8, 2
gate_logits = np.random.randn(num_tokens, num_experts)

selected_experts, weights = top_k_routing(gate_logits, k=top_k)
print(f"Tokens: {num_tokens}, Experts: {num_experts}, Top-K: {top_k}")
print(f"First 5 tokens route to experts:\n{selected_experts[:5]}")
print(f"Weights: {weights[:5].round(3)}")

## 2. Expert Load Distribution & Imbalance

In [ ]:
def compute_load_imbalance(selected_experts: np.ndarray, num_experts: int):
    load = np.bincount(selected_experts.flatten(), minlength=num_experts)
    return load, load.max() / load.mean(), load.std() / load.mean()

load, imbalance, cv = compute_load_imbalance(selected_experts, num_experts)
print(f"Load per expert: {load}")
print(f"Imbalance factor (max/mean): {imbalance:.3f}, CV: {cv:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(num_experts), load, color='steelblue', edgecolor='black')
ax.axhline(load.mean(), color='red', linestyle='--', label=f'Mean={load.mean():.0f}')
ax.set_xlabel('Expert ID'); ax.set_ylabel('Tokens Assigned')
ax.set_title('Expert Load Distribution (Uniform Logits)')
ax.legend(); plt.tight_layout(); plt.show()

## 3. Skewed Routing — Simulating Real-World Imbalance

In [ ]:
skewed_logits = np.random.randn(num_tokens, num_experts)
skewed_logits[:, 0] += 2.0; skewed_logits[:, 1] += 1.5

skewed_experts, _ = top_k_routing(skewed_logits, k=top_k)
skewed_load, skewed_imb, _ = compute_load_imbalance(skewed_experts, num_experts)
print(f"Skewed load: {skewed_load}")
print(f"Imbalance: {skewed_imb:.3f} (was {imbalance:.3f} uniform)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ld, title in zip(axes, [load, skewed_load], ['Uniform', 'Skewed']):
    ax.bar(range(num_experts), ld, color='steelblue' if title=='Uniform' else 'coral', edgecolor='black')
    ax.axhline(ld.mean(), color='red', linestyle='--')
    ax.set_xlabel('Expert ID'); ax.set_ylabel('Tokens')
    ax.set_title(f'{title} Routing'); ax.set_ylim(0, max(skewed_load)*1.1)
plt.tight_layout(); plt.show()

## 4. Double Penalty Calculator

Imbalanced routing causes: (1) **Compute penalty** — all GPUs wait for the slowest expert, (2) **Memory penalty** — buffers sized for worst-case waste capacity on underloaded experts.

In [ ]:
def double_penalty(load: np.ndarray):
    mean_l, max_l, n = load.mean(), load.max(), len(load)
    compute_waste = (max_l - mean_l) / mean_l * 100
    memory_waste = (max_l * n - load.sum()) / load.sum() * 100
    return {'compute_waste_%': compute_waste, 'memory_waste_%': memory_waste,
            'effective_util': mean_l / max_l,
            'total_penalty_factor': (max_l / mean_l) ** 2}

print("=== Uniform ===")
for k, v in double_penalty(load).items(): print(f"  {k}: {v:.3f}")
print("\n=== Skewed ===")
for k, v in double_penalty(skewed_load).items(): print(f"  {k}: {v:.3f}")

## 5. Wide-EP Communication Volume

In Wide Expert Parallelism, tokens dispatch across GPUs via all-to-all. Volume = `tokens × top_k × hidden_dim × 2 × cross_gpu_fraction`.

In [ ]:
def wide_ep_comm(num_tokens, hidden_dim, top_k, num_gpus, dtype_bytes=2):
    tpg = num_tokens // num_gpus
    cross = (num_gpus - 1) / num_gpus
    return tpg * top_k * hidden_dim * dtype_bytes * cross * 2  # dispatch+combine

hidden_dim, batch_tokens = 4096, 4096
gpu_counts = [2, 4, 8, 16, 32, 64]
vols = [wide_ep_comm(batch_tokens, hidden_dim, 2, g) / 1e6 for g in gpu_counts]

for g, v in zip(gpu_counts, vols):
    print(f"GPUs={g:2d}: {v:.1f} MB/GPU per forward pass")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gpu_counts, vols, 'o-', color='darkblue', linewidth=2)
ax.set_xlabel('Number of GPUs'); ax.set_ylabel('Comm Volume/GPU (MB)')
ax.set_title('Wide-EP All-to-All Communication Volume')
ax.set_xscale('log', base=2); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. MoE vs Dense: Cost Comparison at Various GPU Counts

In [ ]:
gpu_cost_hr = 3.0  # $/hr per A100-80GB
configs = {'Mixtral-8x7B (MoE)': (2, 2500), 'Llama-70B (Dense)': (2, 800), 'Llama-13B (Dense)': (1, 4000)}

print(f"{'Model':<25} {'GPUs':>5} {'$/1M tok':>10} {'Tok/s':>7}")
print('-' * 50)
for name, (gpus, tps) in configs.items():
    cost = gpus * gpu_cost_hr / (tps * 3600) * 1e6
    print(f"{name:<25} {gpus:>5} ${cost:>8.4f} {tps:>7}")

# Scaling: MoE has all-to-all overhead, dense scales better
gpu_scale = [1, 2, 4, 8, 16]
moe_eff = [1.0, 1.8, 3.2, 5.5, 9.0]
dense_eff = [1.0, 1.9, 3.6, 6.8, 12.5]

moe_c = [g * gpu_cost_hr / (2500*s*3600)*1e6 for g, s in zip(gpu_scale, moe_eff)]
dense_c = [g * gpu_cost_hr / (800*s*3600)*1e6 for g, s in zip(gpu_scale, dense_eff)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(gpu_scale, moe_c, 'o-', label='Mixtral-8x7B (MoE)', color='coral', lw=2)
ax.plot(gpu_scale, dense_c, 's-', label='Llama-70B (Dense)', color='steelblue', lw=2)
ax.set_xlabel('GPU Count'); ax.set_ylabel('Cost per 1M Tokens ($)')
ax.set_title('MoE vs Dense: Cost Scaling'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("MoE wins at low GPU counts; dense catches up at scale due to all-to-all overhead.")